# VulnSneak — Family Classification Model Training
**Stage 3 of 3: Fine-tuning CodeBERT for Vulnerability Type Classification**

---

## Objective

Train an 8-class sequence classifier on top of `microsoft/codebert-base` to answer
one question for any code snippet already detected as Vulnerable by the Binary Model:

> Which vulnerability type is present in this code?

This model is the second stage of the VulnSneak cascade pipeline.
It only runs on code that the Binary Model has already flagged as Vulnerable.

---

## Inputs and Outputs

| Item | Value |
|---|---|
| Base model | `microsoft/codebert-base` |
| Training data | `data/family_train.jsonl` — 17,007 samples |
| Validation data | `data/family_val.jsonl` — 2,124 samples |
| Test data | `data/family_test.jsonl` — 2,129 samples |
| Labels | 8 vulnerability families |
| Output | `models/family_model/` |

---

## Primary Evaluation Metric

**Macro F1.**

Macro F1 computes F1 independently for each of the 8 classes and then
averages them without weighting by class size. This means every vulnerability
family contributes equally to the final score, regardless of how many samples
it has. This is the correct metric when all vulnerability types matter equally.

---

## Key Difference from Binary Model

The family dataset is **imbalanced** — some vulnerability families have
significantly more samples than others. This notebook applies
**inverse-frequency class weights** during training to prevent the model
from being biased toward the larger families.

---

## Runtime Requirement

This notebook requires a **GPU runtime**.
In Colab: Runtime > Change runtime type > A100 GPU (recommended).


## 1. Environment Setup

In [1]:
import subprocess
subprocess.run([
    "pip", "install", "-q",
    "transformers==4.40.0",
    "datasets==2.19.0",
    "scikit-learn==1.4.2",
    "accelerate==0.30.0",
    "peft==0.10.0",
], check=True)

print("Installation complete.")


Installation complete.


## 2. GPU Verification

In [2]:
import torch

if not torch.cuda.is_available():
    raise EnvironmentError(
        "No GPU detected. "
        "Go to Runtime > Change runtime type and select A100 GPU."
    )

device   = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"Device : {gpu_name}")
print(f"Memory : {gpu_mem:.1f} GB")
print(f"CUDA   : {torch.version.cuda}")


Device : NVIDIA A100-SXM4-80GB
Memory : 85.1 GB
CUDA   : 12.8


## 3. Mount Google Drive

**Expected Drive structure before running this notebook:**

```
MyDrive/
  VulnSneak/
    data/
      family_train.jsonl
      family_val.jsonl
      family_test.jsonl
    models/
      binary_model/   (already trained in Stage 2)
```


In [3]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# Set PROJECT_ROOT to your working directory
# Example (Colab): PROJECT_ROOT = Path("/content/drive/MyDrive/VulnSneak")
# Example (local): PROJECT_ROOT = Path("./VulnSneak")
PROJECT_ROOT = Path("/content/drive/MyDrive/VulnSneak")  # <-- change this
DATA_DIR     = PROJECT_ROOT / "data"
MODEL_DIR    = PROJECT_ROOT / "models" / "family_model"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

required_files = ["family_train.jsonl", "family_val.jsonl", "family_test.jsonl"]
for fname in required_files:
    fpath = DATA_DIR / fname
    assert fpath.exists(), f"File not found: {fpath}"
    print(f"Found: {fpath}")

print("\nAll dataset files verified.")


Mounted at /content/drive
Found: /content/drive/MyDrive/VulnSneak/data/family_train.jsonl
Found: /content/drive/MyDrive/VulnSneak/data/family_val.jsonl
Found: /content/drive/MyDrive/VulnSneak/data/family_test.jsonl

All dataset files verified.


## 4. Configuration

### Notes on key differences from the Binary Model

| Parameter | Binary Model | Family Model | Reason |
|---|---|---|---|
| `num_labels` | 2 | 8 | One class per vulnerability family |
| `max_length` | 256 | 256 | Same — dataset token distribution unchanged |
| `batch_size` | 32 | 32 | Same hardware, same sample size |
| `class_weights` | equal | inverse frequency | Family dataset is imbalanced |
| `metric_for_best` | Vulnerable Recall | Macro F1 | All 8 families must perform equally |


In [4]:
# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME = "microsoft/codebert-base"
NUM_LABELS = 8

FAMILIES = [
    "CSRF",
    "Insecure Cryptography",
    "Insecure Deserialization",
    "OS Command Injection",
    "Path Traversal",
    "SQL Injection",
    "XML Injection",
    "XSS",
]

LABEL2ID = {family: idx for idx, family in enumerate(FAMILIES)}
ID2LABEL = {idx: family for idx, family in enumerate(FAMILIES)}

# ── Tokenization ──────────────────────────────────────────────────────────────
MAX_LENGTH = 256

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE     = 32
LEARNING_RATE  = 2e-5
NUM_EPOCHS     = 5
WARMUP_RATIO   = 0.1
WEIGHT_DECAY   = 0.01
RANDOM_SEED    = 42

# ── Early stopping ────────────────────────────────────────────────────────────
EARLY_STOPPING_PATIENCE = 2

# ── Checkpoint selection metric ───────────────────────────────────────────────
METRIC_FOR_BEST = "eval_macro_f1"

print("Configuration loaded.")
print(f"  Model      : {MODEL_NAME}")
print(f"  Num labels : {NUM_LABELS}")
print(f"  Max length : {MAX_LENGTH}")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  Epochs     : {NUM_EPOCHS}")
print(f"  LR         : {LEARNING_RATE}")
print()
print("Label mapping:")
for family, idx in LABEL2ID.items():
    print(f"  {idx}  ->  {family}")


Configuration loaded.
  Model      : microsoft/codebert-base
  Num labels : 8
  Max length : 256
  Batch size : 32
  Epochs     : 5
  LR         : 2e-05

Label mapping:
  0  ->  CSRF
  1  ->  Insecure Cryptography
  2  ->  Insecure Deserialization
  3  ->  OS Command Injection
  4  ->  Path Traversal
  5  ->  SQL Injection
  6  ->  XML Injection
  7  ->  XSS


## 5. Load Datasets

Each sample contains:
- `code`    — the vulnerable code snippet
- `label`   — the vulnerability family name
- `pair_id` — original pair index for traceability

Only vulnerable code is present in the family dataset.
Safe samples were excluded during `prepare_datasets.ipynb`.


In [5]:
import json
from collections import Counter

def load_jsonl(filepath) -> list[dict]:
    samples = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                samples.append(json.loads(line))
    return samples


train_samples = load_jsonl(DATA_DIR / "family_train.jsonl")
val_samples   = load_jsonl(DATA_DIR / "family_val.jsonl")
test_samples  = load_jsonl(DATA_DIR / "family_test.jsonl")


def print_distribution(name: str, samples: list[dict]) -> None:
    counts = Counter(s["label"] for s in samples)
    total  = sum(counts.values())
    print(f"  {name}")
    for family in FAMILIES:
        count = counts.get(family, 0)
        bar   = "#" * (count // 50)
        print(f"    {family:<35} {count:>5,}  ({count/total*100:>4.1f}%)  {bar}")
    print()


print("Label Distribution")
print("-" * 65)
print_distribution("family_train", train_samples)
print_distribution("family_val",   val_samples)
print_distribution("family_test",  test_samples)


Label Distribution
-----------------------------------------------------------------
  family_train
    CSRF                                2,160  (12.7%)  ###########################################
    Insecure Cryptography               2,364  (13.9%)  ###############################################
    Insecure Deserialization            1,779  (10.5%)  ###################################
    OS Command Injection                2,160  (12.7%)  ###########################################
    Path Traversal                      1,880  (11.1%)  #####################################
    SQL Injection                       2,219  (13.0%)  ############################################
    XML Injection                       1,824  (10.7%)  ####################################
    XSS                                 2,621  (15.4%)  ####################################################

  family_val
    CSRF                                  270  (12.7%)  #####
    Insecure Cryptography      

## 6. Class Weights

The family dataset is imbalanced — SQL Injection and XSS have more samples
than families like Insecure Deserialization or XML Injection.

Without correction, the model would learn to favor the larger families.

**Inverse-frequency class weights** penalize the model more heavily for
mistakes on smaller families, forcing it to pay equal attention to all 8 classes.

The weight for each class is computed as:

```
weight = total_samples / (num_classes * count_of_class)
```

A class with fewer samples gets a higher weight.


In [6]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

train_label_ids = [LABEL2ID[s["label"]] for s in train_samples]

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_LABELS),
    y=np.array(train_label_ids),
)

class_weight_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print("Class Weights (higher = model penalized more for mistakes on this class)")
print("-" * 55)
for idx, weight in enumerate(class_weights):
    family = ID2LABEL[idx]
    count  = train_label_ids.count(idx)
    print(f"  {family:<35} count={count:>5,}  weight={weight:.4f}")


Class Weights (higher = model penalized more for mistakes on this class)
-------------------------------------------------------
  CSRF                                count=2,160  weight=0.9842
  Insecure Cryptography               count=2,364  weight=0.8993
  Insecure Deserialization            count=1,779  weight=1.1950
  OS Command Injection                count=2,160  weight=0.9842
  Path Traversal                      count=1,880  weight=1.1308
  SQL Injection                       count=2,219  weight=0.9580
  XML Injection                       count=1,824  weight=1.1655
  XSS                                 count=2,621  weight=0.8111


## 7. Tokenization

Tokenize the code snippets using the CodeBERT tokenizer.
The dataset and tokenization logic are identical to the Binary Model,
with the only difference being the label mapping (8 classes instead of 2).


In [7]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset
import torch


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")


class FamilyCodeDataset(Dataset):
    """
    PyTorch Dataset for vulnerability family classification.

    Each sample contains a vulnerable code snippet and its
    corresponding family label as an integer ID.
    """

    def __init__(self, samples: list[dict], tokenizer, max_length: int):
        self.samples    = samples
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> dict:
        sample  = self.samples[idx]
        encoded = self.tokenizer(
            sample["code"],
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids":      encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels":         torch.tensor(LABEL2ID[sample["label"]], dtype=torch.long),
        }


train_dataset = FamilyCodeDataset(train_samples, tokenizer, MAX_LENGTH)
val_dataset   = FamilyCodeDataset(val_samples,   tokenizer, MAX_LENGTH)
test_dataset  = FamilyCodeDataset(test_samples,  tokenizer, MAX_LENGTH)

print(f"\nDataset sizes")
print(f"  Train : {len(train_dataset):,}")
print(f"  Val   : {len(val_dataset):,}")
print(f"  Test  : {len(test_dataset):,}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Tokenizer loaded: microsoft/codebert-base

Dataset sizes
  Train : 17,007
  Val   : 2,124
  Test  : 2,129


## 8. Load Model

Load `microsoft/codebert-base` with an 8-class sequence classification head.

This is a separate model from the Binary Model trained in Stage 2.
Both start from the same CodeBERT weights, but each is fine-tuned
independently on a different task with different decision boundaries.


In [8]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

model = model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model loaded: {MODEL_NAME}")
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded: microsoft/codebert-base
Total parameters     : 124,651,784
Trainable parameters : 124,651,784


## 9. Custom Trainer with Class Weights

The standard Hugging Face Trainer does not support class weights directly.
A subclass is created that overrides the `compute_loss` method to apply
the inverse-frequency weights computed in Cell 6.

This is the critical difference between the Binary Model notebook
and this notebook.


In [9]:
from transformers import Trainer
import torch.nn as nn


class WeightedTrainer(Trainer):
    """
    Trainer subclass that applies class weights to the cross-entropy loss.

    This forces the model to pay equal attention to all 8 vulnerability
    families regardless of their sample counts in the training set.
    """

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits

        loss_fn = nn.CrossEntropyLoss(weight=class_weight_tensor)
        loss    = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss


print("WeightedTrainer defined.")


WeightedTrainer defined.


## 10. Evaluation Metrics

Metrics reported after every epoch:

| Metric | Description |
|---|---|
| `macro_f1` | Primary metric — F1 averaged equally across all 8 families |
| `weighted_f1` | F1 weighted by class size — secondary reference |
| `accuracy` | Overall correctness |
| `per_class_f1` | F1 for each individual family — for diagnosing weak classes |

The best checkpoint is selected based on **Macro F1** on the validation set.


In [10]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
)


def compute_metrics(eval_pred) -> dict:
    """
    Custom metric function for the Hugging Face Trainer.

    Returns macro F1, weighted F1, accuracy, and per-class F1
    for all 8 vulnerability families.
    """
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=-1)

    accuracy     = accuracy_score(labels, predictions)
    macro_f1     = f1_score(labels, predictions, average="macro",    zero_division=0)
    weighted_f1  = f1_score(labels, predictions, average="weighted", zero_division=0)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        labels=list(range(NUM_LABELS)),
        zero_division=0,
    )

    metrics = {
        "accuracy"    : round(accuracy, 4),
        "macro_f1"    : round(macro_f1, 4),
        "weighted_f1" : round(weighted_f1, 4),
    }

    for idx in range(NUM_LABELS):
        family_key = ID2LABEL[idx].lower().replace(" ", "_")
        metrics[f"{family_key}_f1"]        = round(f1[idx], 4)
        metrics[f"{family_key}_precision"] = round(precision[idx], 4)
        metrics[f"{family_key}_recall"]    = round(recall[idx], 4)

    return metrics


print("Metric function defined.")
print(f"Checkpoint selection : {METRIC_FOR_BEST} (higher is better)")


Metric function defined.
Checkpoint selection : eval_macro_f1 (higher is better)


## 11. Training Arguments

Identical structure to the Binary Model, with two differences:
- `metric_for_best_model` is set to `eval_macro_f1`
- The WeightedTrainer is used instead of the standard Trainer


In [11]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / "checkpoints"),

    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,

    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model=METRIC_FOR_BEST,
    greater_is_better=True,

    logging_dir=str(MODEL_DIR / "logs"),
    logging_strategy="steps",
    logging_steps=100,
    report_to="none",

    seed=RANDOM_SEED,
    fp16=True,
)

print("Training arguments configured.")


Training arguments configured.


## 12. Build Trainer


In [12]:
from transformers import EarlyStoppingCallback

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE
        )
    ],
)

print("Trainer ready.")
print(f"  Training samples   : {len(train_dataset):,}")
print(f"  Validation samples : {len(val_dataset):,}")
print(f"  Steps per epoch    : {len(train_dataset) // BATCH_SIZE:,}")


Trainer ready.
  Training samples   : 17,007
  Validation samples : 2,124
  Steps per epoch    : 531


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


## 13. Train

The family dataset is half the size of the binary dataset (17K vs 34K samples),
so training will be faster.

Estimated time on A100 GPU: **8 to 12 minutes** for 5 epochs.


In [13]:
import time

print("Starting training...")
print("-" * 50)

start_time   = time.time()
train_result = trainer.train()
elapsed      = time.time() - start_time

print("-" * 50)
print(f"Training complete in {elapsed / 60:.1f} minutes.")
print(f"  Final train loss : {train_result.training_loss:.4f}")


Starting training...
--------------------------------------------------


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Csrf F1,Csrf Precision,Csrf Recall,Insecure Cryptography F1,Insecure Cryptography Precision,Insecure Cryptography Recall,Insecure Deserialization F1,Insecure Deserialization Precision,Insecure Deserialization Recall,Os Command Injection F1,Os Command Injection Precision,Os Command Injection Recall,Path Traversal F1,Path Traversal Precision,Path Traversal Recall,Sql Injection F1,Sql Injection Precision,Sql Injection Recall,Xml Injection F1,Xml Injection Precision,Xml Injection Recall,Xss F1,Xss Precision,Xss Recall
1,0.032100,0.020419,0.996200,0.995900,0.996200,0.998200,0.996300,1.000000,0.998300,1.000000,0.996600,0.988800,0.982200,0.995500,1.000000,1.000000,1.000000,0.989200,1.000000,0.978700,1.000000,1.000000,1.000000,0.995600,0.991300,1.000000,0.996900,0.996900,0.996900
2,0.007200,0.008831,0.998600,0.998600,0.998600,0.994500,0.989000,1.000000,0.998300,1.000000,0.996600,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.997900,1.000000,0.995700,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.998500,1.000000,0.996900
3,0.005800,0.001287,0.999500,0.999600,0.999500,0.998200,0.996300,1.000000,0.998300,1.000000,0.996600,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
4,0.001100,0.001046,0.999500,0.999600,0.999500,0.998200,0.996300,1.000000,0.998300,1.000000,0.996600,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
5,0.000600,0.001050,0.999500,0.999600,0.999500,0.998200,0.996300,1.000000,0.998300,1.000000,0.996600,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


--------------------------------------------------
Training complete in 4.6 minutes.
  Final train loss : 0.1138


## 14. Save Best Model

Save the best checkpoint (selected by Macro F1) to `models/family_model/best/`.


In [14]:
best_model_path = MODEL_DIR / "best"
best_model_path.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(best_model_path))
tokenizer.save_pretrained(str(best_model_path))

print(f"Best model saved to: {best_model_path}")
print("Contents:")
for f in sorted(best_model_path.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<35} {size_kb:>8.1f} KB")


Best model saved to: /content/drive/MyDrive/VulnSneak/models/family_model/best
Contents:
  config.json                              1.1 KB
  merges.txt                             445.6 KB
  model.safetensors                   486944.6 KB
  special_tokens_map.json                  0.9 KB
  tokenizer.json                        2059.5 KB
  tokenizer_config.json                    1.2 KB
  training_args.bin                        5.3 KB
  vocab.json                             779.6 KB


## 15. Evaluate on Test Set

Run the best model on the held-out test set and report all metrics.

The most important number to report is **Macro F1**.
Per-class F1 identifies which vulnerability families are hardest to classify.


In [15]:
import numpy as np

print("Evaluating on test set...")
test_output = trainer.predict(test_dataset)
metrics     = test_output.metrics

print("\nTest Set Results")
print("=" * 50)
print(f"  Accuracy     : {metrics['test_accuracy']:.4f}")
print(f"  Macro F1     : {metrics['test_macro_f1']:.4f}  <- primary metric")
print(f"  Weighted F1  : {metrics['test_weighted_f1']:.4f}")
print()
print(f"  {'Family':<35} {'F1':>6}  {'Precision':>9}  {'Recall':>6}")
print("-" * 65)
for idx in range(NUM_LABELS):
    family     = ID2LABEL[idx]
    family_key = family.lower().replace(" ", "_")
    f1         = metrics.get(f"test_{family_key}_f1", 0)
    prec       = metrics.get(f"test_{family_key}_precision", 0)
    rec        = metrics.get(f"test_{family_key}_recall", 0)
    print(f"  {family:<35} {f1:>6.4f}  {prec:>9.4f}  {rec:>6.4f}")


Evaluating on test set...



Test Set Results
  Accuracy     : 0.9981
  Macro F1     : 0.9980  <- primary metric
  Weighted F1  : 0.9981

  Family                                  F1  Precision  Recall
-----------------------------------------------------------------
  CSRF                                0.9963     0.9963  0.9963
  Insecure Cryptography               0.9966     0.9966  0.9966
  Insecure Deserialization            0.9955     1.0000  0.9910
  OS Command Injection                1.0000     1.0000  1.0000
  Path Traversal                      0.9958     0.9916  1.0000
  SQL Injection                       1.0000     1.0000  1.0000
  XML Injection                       1.0000     1.0000  1.0000
  XSS                                 1.0000     1.0000  1.0000


## 16. Confusion Matrix

The confusion matrix shows which families are being confused with each other.

A high value off the diagonal (row i, column j where i ≠ j) means
the model is frequently predicting family j when the correct answer is family i.
This is useful for understanding systematic misclassification patterns.


In [16]:
from sklearn.metrics import confusion_matrix

predictions = np.argmax(test_output.predictions, axis=-1)
true_labels = test_output.label_ids

cm = confusion_matrix(true_labels, predictions, labels=list(range(NUM_LABELS)))

# Print confusion matrix
header = "".join(f"{i:>5}" for i in range(NUM_LABELS))
print(f"Confusion Matrix  (rows=actual, cols=predicted)")
print(f"Labels: { {i: f[:4] for i, f in ID2LABEL.items()} }")
print()
print(f"     {header}")
print("    " + "-" * (NUM_LABELS * 5 + 2))
for i, row in enumerate(cm):
    row_str = "".join(f"{v:>5}" for v in row)
    print(f"  {i} |{row_str}  <- {ID2LABEL[i]}")


Confusion Matrix  (rows=actual, cols=predicted)
Labels: {0: 'CSRF', 1: 'Inse', 2: 'Inse', 3: 'OS C', 4: 'Path', 5: 'SQL ', 6: 'XML ', 7: 'XSS'}

         0    1    2    3    4    5    6    7
    ------------------------------------------
  0 |  269    0    0    0    1    0    0    0  <- CSRF
  1 |    1  295    0    0    0    0    0    0  <- Insecure Cryptography
  2 |    0    1  221    0    1    0    0    0  <- Insecure Deserialization
  3 |    0    0    0  270    0    0    0    0  <- OS Command Injection
  4 |    0    0    0    0  235    0    0    0  <- Path Traversal
  5 |    0    0    0    0    0  278    0    0  <- SQL Injection
  6 |    0    0    0    0    0    0  228    0  <- XML Injection
  7 |    0    0    0    0    0    0    0  329  <- XSS


## 17. Save Evaluation Results


In [17]:
import json
from datetime import datetime

per_class = {}
for idx in range(NUM_LABELS):
    family     = ID2LABEL[idx]
    family_key = family.lower().replace(" ", "_")
    per_class[family] = {
        "f1"        : metrics.get(f"test_{family_key}_f1", 0),
        "precision" : metrics.get(f"test_{family_key}_precision", 0),
        "recall"    : metrics.get(f"test_{family_key}_recall", 0),
    }

results = {
    "model"         : MODEL_NAME,
    "max_length"    : MAX_LENGTH,
    "batch_size"    : BATCH_SIZE,
    "learning_rate" : LEARNING_RATE,
    "num_epochs"    : NUM_EPOCHS,
    "class_weights" : {ID2LABEL[i]: round(w, 4) for i, w in enumerate(class_weights)},
    "evaluated_at"  : datetime.now().isoformat(),
    "test_metrics"  : {
        "accuracy"    : metrics["test_accuracy"],
        "macro_f1"    : metrics["test_macro_f1"],
        "weighted_f1" : metrics["test_weighted_f1"],
    },
    "per_class" : per_class,
}

results_path = MODEL_DIR / "test_results.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print(f"Results saved to: {results_path}")


Results saved to: /content/drive/MyDrive/VulnSneak/models/family_model/test_results.json


## 18. Summary

Training is complete. The outputs of this notebook are:

| Output | Location |
|---|---|
| Best model weights | `models/family_model/best/` |
| Training checkpoints | `models/family_model/checkpoints/` |
| Evaluation results | `models/family_model/test_results.json` |

---

## Pipeline Status

| Stage | Model | Status |
|---|---|---|
| Stage 2 | Binary Model (Safe / Vulnerable) | Complete |
| Stage 3 | Family Model (8 vulnerability types) | Complete |

---

## Honest Caveat

The same limitation from the Binary Model applies here:
all training samples are **patched versions** of vulnerable code,
not independently written examples. Test metrics likely overestimate
real-world performance and should be presented with this caveat
in the graduation project.

---

## Next Step

Open `inference.ipynb` to connect both models into a single cascade
pipeline and run end-to-end vulnerability detection on real code files.
